In [12]:
# Import necessary libraries
import numpy as np
import pandas as pd

# Load the dataset from 'finaldiabetes.csv' into a pandas DataFrame
data = pd.read_csv('finaldiabetes.csv')

# Display all column names to get an overview of the dataset
print(data.columns)

Index(['race', 'gender', 'age', 'admission_type_id',
       'discharge_disposition_id', 'admission_source_id', 'time_in_hospital',
       'num_lab_procedures', 'num_procedures', 'num_medications',
       'number_outpatient', 'number_emergency', 'number_inpatient', 'diag_1',
       'diag_2', 'diag_3', 'number_diagnoses', 'metformin', 'repaglinide',
       'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide',
       'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone',
       'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide',
       'examide', 'citoglipton', 'insulin', 'glyburide-metformin',
       'glipizide-metformin', 'glimepiride-pioglitazone',
       'metformin-rosiglitazone', 'metformin-pioglitazone', 'change',
       'diabetesMed', 'readmitted', 'readmitted_binary'],
      dtype='object')


### Overall Readmission Rate Analysis

This section calculates the overall readmission rate from the `readmitted_binary` column and computes its 95% bootstrap confidence interval to estimate the precision of this rate.

In [14]:
# Define a function to calculate bootstrap confidence intervals
def bootstrap_ci(series, n_boot=1000):
    # List to store means from bootstrap samples
    boot_means = []

    # Perform bootstrapping: resample with replacement and calculate mean for each sample
    for _ in range(n_boot):
        # Sample with replacement (frac=1 means sample the entire size of the original series)
        sample = series.sample(frac=1, replace=True)
        boot_means.append(sample.mean())

    # Calculate the 2.5th and 97.5th percentiles to get the 95% confidence interval
    lower = np.percentile(boot_means, 2.5)
    upper = np.percentile(boot_means, 97.5)

    return lower, upper

In [16]:
# Calculate the 95% bootstrap confidence interval for the overall readmission rate
overall_ci = bootstrap_ci(data['readmitted_binary'])

# Print the overall readmission rate and its confidence interval
print("Overall readmission rate:", data['readmitted_binary'].mean())
print("95% CI:", overall_ci)

Overall readmission rate: 0.4608808442898414
95% CI: (np.float64(0.4576867519603797), np.float64(0.4638975689326494))


The estimated overall readmission rate is approximately 46.1%. The 95% bootstrap confidence interval of (45.8%, 46.4%) means that if we were to repeat this sampling and analysis many times, 95% of the confidence intervals constructed would contain the true overall readmission rate for the entire population.

### Hospital Stay Grouping and Analysis

This section categorizes patients into 'Shorter hospital stay' and 'Longer hospital stay' groups based on the median `time_in_hospital`. It then analyzes the readmission rates and their confidence intervals for each group.

In [6]:
# create group
data['hospital_stay_group'] = np.where(
    data['time_in_hospital'] > data['time_in_hospital'].median(),
    'Longer hospital stay',
    'Shorter hospital stay'
)

# check groups
data['hospital_stay_group'].value_counts()

,count
hospital_stay_group,
Shorter hospital stay,63112
Longer hospital stay,38654


In [18]:
# Create a new column 'hospital_stay_group' based on the median 'time_in_hospital'
# Patients with time_in_hospital greater than the median are classified as 'Longer hospital stay',
# otherwise, they are classified as 'Shorter hospital stay'
data['hospital_stay_group'] = np.where(
    data['time_in_hospital'] > data['time_in_hospital'].median(),
    'Longer hospital stay',
    'Shorter hospital stay'
)

# Check the distribution of patients in each newly created group
data['hospital_stay_group'].value_counts()

,count
hospital_stay_group,
Shorter hospital stay,63112
Longer hospital stay,38654


### Analysis of Difference in Readmission Rates

This section calculates the difference in readmission rates between the 'Longer hospital stay' and 'Shorter hospital stay' groups and computes a 95% bootstrap confidence interval for this difference. This helps determine if the observed difference is statistically significant.

In [19]:
# Initialize a list to store results for each hospital stay group
group_results = []

# Iterate through each hospital stay group
for group, group_data in data.groupby('hospital_stay_group'):
    # Calculate the bootstrap confidence interval for the 'readmitted_binary' column within the current group
    ci = bootstrap_ci(group_data['readmitted_binary'])

    # Append the group's readmission rate and its confidence interval to the results list
    group_results.append({
        'Group': group,
        'Readmission Rate': group_data['readmitted_binary'].mean(),
        'CI Lower': ci[0],
        'CI Upper': ci[1]
    })

# Convert the list of results into a pandas DataFrame for better readability and presentation
pd.DataFrame(group_results)

,Group,Readmission Rate,CI Lower,CI Upper
0,Longer hospital stay,0.490273,0.485201,0.495524
1,Shorter hospital stay,0.442879,0.439029,0.446683


###Readmission Rates by Hospital Stay Group:

Longer hospital stay: For patients with a longer hospital stay, the estimated readmission rate is approximately 49.0%. We are 95% confident that the true readmission rate for patients with longer hospital stays is between 48.5% and 49.6%.

Shorter hospital stay: For patients with a shorter hospital stay, the estimated readmission rate is approximately 44.3%. We are 95% confident that the true readmission rate for patients with shorter hospital stays is between 43.9% and 44.7%.

In [20]:
# Isolate the 'readmitted_binary' data for 'Longer hospital stay' group
group_long = data[data['hospital_stay_group'] == 'Longer hospital stay']['readmitted_binary']
# Isolate the 'readmitted_binary' data for 'Shorter hospital stay' group
group_short = data[data['hospital_stay_group'] == 'Shorter hospital stay']['readmitted_binary']

# Initialize a list to store the differences of means from bootstrap samples
boot_diffs = []

# Perform bootstrapping for the difference in means
for _ in range(1000):
    # Sample with replacement from both groups
    sample_long = group_long.sample(frac=1, replace=True)
    sample_short = group_short.sample(frac=1, replace=True)

    # Calculate the difference in means for the current bootstrap sample
    boot_diffs.append(sample_long.mean() - sample_short.mean())

# Calculate the 2.5th and 97.5th percentiles of the bootstrap differences
# to get the 95% confidence interval for the difference
diff_lower = np.percentile(boot_diffs, 2.5)
diff_upper = np.percentile(boot_diffs, 97.5)

# Print the observed difference in readmission rates between the two groups
print("Difference in readmission rates:", group_long.mean() - group_short.mean())
# Print the 95% confidence interval for this difference
print("95% CI for difference:", (diff_lower, diff_upper))

Difference in readmission rates: 0.047393349888338876
95% CI for difference: (np.float64(0.04117219815583197), np.float64(0.05363480638227925))


###Difference in Readmission Rates

The difference in readmission rates between the 'Longer hospital stay' group and the 'Shorter hospital stay' group is approximately 4.7 percentage points (49.0% - 44.3%). The 95% bootstrap confidence interval for this difference is (4.1%, 5.4%). Since this entire interval is above zero, it strongly suggests that the difference is statistically significant. In other words, patients with longer hospital stays have a statistically higher readmission rate than those with shorter hospital stays, and we are 95% confident that this difference lies between 4.1 and 5.4 percentage points in the population.

### Uncertainty in our context

In this context, uncertainty refers to the inherent variability and imprecision in our estimates because we are drawing conclusions about a larger population based on a finite sample of data. While we can calculate a specific readmission rate from our dataset, this rate is just an estimate of the true readmission rate in the entire patient population.

The 95% confidence intervals are our way of quantifying this uncertainty. They provide a range within which the true population parameter (be it the overall rate, a group's rate, or the difference between groups' rates) is likely to fall. A narrower confidence interval indicates greater precision (less uncertainty) in our estimate, while a wider interval suggests less precision (more uncertainty).

Given that our confidence intervals are all less than 2 percentage points apart, there is relatively little variability in our bootstrap estimates, which suggests that our sample provides a fairly precise estimate of the true readmission rate. In other words, if we were to repeatedly draw similar samples from the same population, we would expect the estimated readmission rate to fall within a very similar range each time. This increases our confidence that the observed rate is not heavily influenced by random sampling noise, and that it is a stable reflection of the underlying patient population.